In [ ]:
#use "../lib/nb.ml";;
load_css ();;
run_notebook "Types.ipynb"

# Drawing Finite State Machines

This notebook provides functions that convert finite state machines into strings and into graphs.

## Type Checking

The functions in this notebook carry *type annotations*.  In OCaml, these annotations are checked by the
compiler before a cell is executed, so there is no need for a separate type checker like *basedpyright*.

In Python, the functions in this notebook work both for deterministic and for non-deterministic finite state machines,
since the Python type `FSM` describes both kinds of machines.  In OCaml, *Dfa*s and *Nfa*s have different types,
which are defined in the notebook `Types.ipynb`.  Therefore, there are separate functions for *Dfa*s
(`dfa2string` and `dfa2dot`) and for *Nfa*s (`nfa2string` and `nfa2dot`).  In both types, the states can be
arbitrary values: the type parameter `'q` is the type of the states.

Python can convert every value into a string.  OCaml has no such function, since types do not exist at runtime.
Therefore, the functions that have to print states take a function `show` as an additional argument, which
converts a state into a string.  For example, if the states are integers, `show` is `string_of_int`, and if they
are sets of integers, `show` is `FSet.to_string string_of_int`.  Since `show` is a *labelled argument*, it is
passed as `~show:string_of_int`.

In [ ]:
#show_type dfa;;
#show_type nfa

The function `dfa2string` converts the given <span style="font-variant:small-caps;">Dfa</span> into a string.

**Function `dfa2string ~show dfa`**
- *Input:* `dfa` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A string that describes `dfa`.  The states are renamed to `S0`, `S1`, ... .

In [ ]:
let dfa2string ~(show : 'q -> string) (dfa : 'q dfa) : string =
  let ({ states; sigma; delta; start = q0; accepting = final } : 'q dfa) = dfa in
  let statesToNames : ('q, string) Dict.t =
    Dict.of_list (List.mapi (fun n q -> (q, Printf.sprintf "S%d" n)) (FSet.to_list states))
  in
  let name q = Dict.find q statesToNames in
  let n = FSet.cardinal states in
  let result = Buffer.create 256 in
  let add s = Buffer.add_string result s in
  add (Printf.sprintf "states: {S0, ..., S%d}\n\n" (n - 1));
  add (Printf.sprintf "start state: %s\n\n" (name q0));
  add "state encoding:\n";
  FSet.iter (fun q -> add (Printf.sprintf "%s = %s\n" (name q) (show q))) states;
  add "\ntransitions:\n";
  FSet.iter
    (fun q ->
      FSet.iter
        (fun c ->
          match Dict.find_opt (q, c) delta with
          | Some p -> add (Printf.sprintf "delta(%s, %c) = %s\n" (name q) c (name p))
          | None -> ())
        sigma)
    states;
  add "\nset of accepting states: {";
  add (String.concat ", " (List.map name (FSet.to_list final)));
  add "}\n";
  Buffer.contents result

In [ ]:
load_lib "graphviz.ml"

The function `dfa2dot` converts the given <span style="font-variant:small-caps;">Dfa</span> into a graph in dot-format.
The graph is built with the module `Graphviz` from the file `../lib/graphviz.ml`, which replaces the Python package
`graphviz`.  A graph `g` is displayed with `Graphviz.display g`.

**Function `dfa2dot dfa`**
- *Input:* `dfa` is a <span style="font-variant:small-caps;">Dfa</span>.
- *Output:* A pair consisting of a graph of type `Graphviz.t` of `dfa` and a dictionary that maps the states of `dfa` to their names in the graph.

In [ ]:
let dfa2dot (dfa : 'q dfa) : Graphviz.t * ('q, string) Dict.t =
  let ({ states; sigma; delta; start = q0; accepting = final } : 'q dfa) = dfa in
  let dot = Graphviz.digraph "Deterministic FSM" in
  Graphviz.graph_attr dot "rankdir" "LR";
  (* assign a name to every state *)
  let statesToNames : ('q, string) Dict.t =
    Dict.of_list (List.mapi (fun n q -> (q, Printf.sprintf "S%d" n)) (FSet.to_list states))
  in
  let name q = Dict.find q statesToNames in
  let startName = name q0 in
  Graphviz.node dot ~attrs:[ ("label", ""); ("width", "0.1"); ("height", "0.1"); ("style", "filled"); ("color", "blue") ] "1";
  Graphviz.edge dot "1" startName;
  FSet.iter
    (fun q ->
      if FSet.mem q final then Graphviz.node dot ~attrs:[ ("peripheries", "2") ] (name q)
      else Graphviz.node dot (name q))
    states;
  FSet.iter
    (fun q ->
      FSet.iter
        (fun c ->
          match Dict.find_opt (q, c) delta with
          | Some p -> Graphviz.edge dot ~attrs:[ ("label", String.make 1 c) ] (name q) (name p)
          | None -> ())
        sigma)
    states;
  (dot, statesToNames)

The function `nfa2string` converts a non-deterministic finite state machine `nfa` into a string.

**Function `nfa2string ~show nfa`**
- *Input:* `nfa` is an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* A string that describes `nfa`.

In [ ]:
let nfa2string ~(show : 'q -> string) (nfa : 'q nfa) : string =
  let ({ states; sigma; delta; start = q0; accepting = final } : 'q nfa) = nfa in
  let result = Buffer.create 256 in
  let add s = Buffer.add_string result s in
  add (Printf.sprintf "states: %s\n\n" (FSet.to_string show states));
  add (Printf.sprintf "start state: %s\n\n" (show q0));
  add "transitions:\n";
  FSet.iter
    (fun q ->
      FSet.iter
        (fun c ->
          match Dict.find_opt (q, Sym c) delta with
          | Some s -> FSet.iter (fun p -> add (Printf.sprintf "[%s, %c] |-> %s\n" (show q) c (show p))) s
          | None -> ())
        sigma;
      match Dict.find_opt (q, Eps) delta with
      | Some s -> FSet.iter (fun p -> add (Printf.sprintf "[%s, 𝜀] |-> %s\n" (show q) (show p))) s
      | None -> ())
    states;
  add (Printf.sprintf "\nset of accepting states: %s\n" (FSet.to_string show final));
  Buffer.contents result

The function `nfa2dot` takes a non-deterministic finite state machine and converts it 
into a a dot graph.

**Function `nfa2dot ~show nfa`**
- *Input:* `nfa` is an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* A graph of type `Graphviz.t` of `nfa`.

In [ ]:
let nfa2dot ~(show : 'q -> string) (nfa : 'q nfa) : Graphviz.t =
  let ({ states; sigma; delta; start = q0; accepting = final } : 'q nfa) = nfa in
  let startName = show q0 in
  let dot = Graphviz.digraph "Non-Deterministic FSM" in
  Graphviz.graph_attr dot "rankdir" "LR";
  Graphviz.node dot ~attrs:[ ("label", ""); ("width", "0.1"); ("height", "0.1"); ("style", "filled"); ("color", "blue") ] "0";
  Graphviz.edge dot "0" startName;
  FSet.iter
    (fun q ->
      if FSet.mem q final then Graphviz.node dot ~attrs:[ ("peripheries", "2") ] (show q)
      else Graphviz.node dot (show q))
    states;
  FSet.iter
    (fun q ->
      match Dict.find_opt (q, Eps) delta with
      | Some s -> FSet.iter (fun p -> Graphviz.edge dot ~attrs:[ ("label", "𝜀"); ("weight", "0.1") ] (show q) (show p)) s
      | None -> ())
    states;
  FSet.iter
    (fun q ->
      FSet.iter
        (fun c ->
          match Dict.find_opt (q, Sym c) delta with
          | Some s ->
              FSet.iter (fun p -> Graphviz.edge dot ~attrs:[ ("label", String.make 1 c); ("weight", "10") ] (show q) (show p)) s
          | None -> ())
        sigma)
    states;
  dot